# Model Context Protocol (MCP)

We can use any LLM with MCP. In this repository we will use `llama3.2:1b` and `llama3.2:3b`

You can also download other models such as `llama3.2:8b` (if your machine supports)

| Model | Command |
| --- | --- |
| llama 3.2:1b | `ollama pull llama3.2:1b` |
| llama 3.2:3b | `ollama pull llama3.2:3b` |
| llama 3.2:8b | `ollama pull llama3.2:8b` |

In [107]:
# --- 1. Imports --------------------------------------------------------------

import os
import sys
import ollama
from pprint import pprint

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


In [108]:
# --- 2. Model configuration --------------------------------------------------
LARGE_MODEL='llama3.2:3b' # or llama3.2:1b for faster but less accurate results
SMALL_MODEL='llama3.2:1b' # or llama3.2:3b for better but slower results

We have a folder `mcp_servers`, inside this folder we have our mcp tool written under different python scripts. These scripts are also called servers in some cases (when these are running on a remote server waiting for a client connection). Because MCP tools can be used as plug and play unlike agents, these scripts can run on remote servers

First, we have a very simple mcp tool which is a tool to add two numbers under the script `the_math_server.py`.
We will use this tool first just to understand how the mcp works with an LLM (in this case llama3.2)


In [109]:
# Configuration for an MCP Server (Example: a local python script)
# If you don't have one, you can use a simple 'hello-world' MCP server script

# --- 3. MCP server paths -----------------------------------------------------

# Math server (works)
MCP_MATH_SERVER = os.path.abspath("./mcp_servers/the_math_server.py")

# Tools server (web search + file reader)
MCP_TOOLS_SERVER = os.path.abspath("./mcp_servers/mcp_tools_server.py")

# Always run Python scripts via the Python interpreter
MCP_SERVER_PATH = sys.executable


In [110]:
# --- 4. Generic MCP runner ---------------------------------------------------

async def run_mcp_query(server_script, user_prompt, model):
    """
    Launches an MCP server (Python script), lists tools,
    lets the LLM choose a tool, executes it, and returns the final answer.
    """

    server_params = StdioServerParameters(
        command=MCP_SERVER_PATH,
        args=[server_script]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:

            # Handshake
            await session.initialize()

            # List tools
            tools = await session.list_tools()

            # Convert MCP tools → Ollama tool format
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in tools.tools
            ]

            # Ask LLM what to do
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": user_prompt}],
                tools=ollama_tools,
            )

            # If LLM calls a tool
            if response.get("message", {}).get("tool_calls"):
                for tool_call in response["message"]["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    tool_args = tool_call["function"]["arguments"]

                    print(f"🛠️ Calling tool: {tool_name} with {tool_args}")

                    result = await session.call_tool(tool_name, tool_args)
                    print(f"✅ Tool Result: {result.content}")

                    # Final answer
                    final = ollama.chat(
                        model=model,
                        messages=[
                            {"role": "user", "content": user_prompt},
                            response["message"],
                            {
                                "role": "tool",
                                "content": str(result.content),
                                "name": tool_name,
                            },
                        ],
                    )
                    return final["message"]["content"]

            # If no tool was used
            return response["message"]["content"]


### Code Explanation

`async def run_mcp_query(user_prompt, MODEL):` 
- The word async (asynchronous) just means your computer won't freeze up while it waits for the process to finish. It can go do other chores, like a person putting a phone on speaker while waiting on hold.
- The Layman Translation: "I am preparing to ask my assistant a question (the user_prompt), and I'm willing to wait patiently for the answer."

`async with stdio_client(server_params) as (read, write):`
- This step builds the actual physical/digital pipe between your main program and the hidden tool server. The server_params are like the phone number or the address of the office.
- The Layman Translation: "I am plugging a secure phone line directly into the assistant's office. I have an earpiece to listen (read) and a microphone to speak (write)."

`async with ClientSession(read, write) as session:`
- Just having a wire isn't enough; the computers need rules on how to format their messages. A ClientSession wraps the raw wire in a polite protocol (like agreeing to say "Over" when you finish speaking on a walkie-talkie).
- The Layman Translation: "Now that the wire is connected, we are agreeing to speak the same language so we don't talk over each other."

`await session.initialize()`
- Before you can ask your actual question or ask the server to run tools, you have to do a digital handshake. This proves the server is awake, functioning, and ready to accept commands.
- The Layman Translation: "I am saying 'Hello, are you there and ready to work?' and I will wait (await) until you say 'Yes, I am ready!'"

Let us now try to use our MCP tool

In [111]:
query = "What is 5 + 5?"
result = await run_mcp_query(MCP_MATH_SERVER, query, SMALL_MODEL)
print("LLM Final Response:", result)


[02/23/26 13:59:02] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=338807;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=157028;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

🛠️ Calling tool: add_numbers with {'a': '5', 'b': '5'}
✅ Tool Result: [TextContent(type='text', text='10', annotations=None, meta=None)]


                    INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=690643;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=387189;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

LLM Final Response: I'm sorry, but I cannot display the result of a calculation. Would you like to try another question?


In [112]:
query = "Who won the FIFA World Cup in 2026?"
result = await run_mcp_query(MCP_TOOLS_SERVER, query, SMALL_MODEL)
print("LLM Final Response:", result)


[02/23/26 13:59:08] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=631019;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=177372;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

🛠️ Calling tool: search_web with {'query': 'FIFA World Cup winner 2026'}
✅ Tool Result: [TextContent(type='text', text='EA足球游戏《FIFA 23》已于昨日正式发售，目前该作在Steam的口碑并不太好，为多半差评，2775篇评测中仅有…\n---\n25. Jan. 2023 · 对我来说入坑那一部就是最好玩的一部——FIFA15mobile。 永远忘不了童年在iPad2上发疯了似的玩这款游戏，虽然上小学那会儿只能打过半职业 …\n---\n一时间有点愕然，不过转念一想，这也算是意料之外，情理之中的决定。 从CM99开始，半路分家变成FM，到如今，《足球经理》系列也有26年了，和其他体育竞 …', annotations=None, meta=None)]


[02/23/26 13:59:09] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=159510;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=242364;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

LLM Final Response: I'm not aware of any information about the winner of the 2026 FIFA World Cup. The tournament is scheduled to take place in 2026, but I don't have any updates or news about its outcome yet. Would you like me to check for any other information about the tournament?


In [113]:
print("SERVER STARTING...")

from mcp.server.fastmcp import FastMCP
from duckduckgo_search import DDGS

print("IMPORTS OK")

mcp = FastMCP("SuperServer")

print("MCP CREATED")


SERVER STARTING...
IMPORTS OK
MCP CREATED


You will see above that the `SMALL_MODEL` fails to give the answer to a very simple question. Though the process successfully uses the `Math` MCP tool and the result was also calculated correctly as can be seen in the 2nd line of the output under `text='10'`.

Lets now try the same query with the `LARGE_MODEL`

In [114]:
result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL)
print(f"LLM Final Response: {result}")

  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/2623108008.py", line 1, in <module>
  |     result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL)
  |              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1811137468.py", line 14, in run_mcp_query
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |   

## Run a more complex MCP servers

There is another server file `mcp_tools_server.py` inside the `mcp_servers` folder. 

This mcp server scripts consists of two tools.
1. Web Search Tool - using DuckDuckGo 
2. Reading a local file Tool

In [124]:
# trouble shooting, with this new setup: Instead of pointing to the .py file directly, 
# point to the Python interpreter and pass the script as an argument.
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_query(server_path, query, model):
    server_params = StdioServerParameters(
        command=server_path,      # <-- python interpreter
        args=MCP_SERVER_ARGS      # <-- your script
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            response = await session.call_tool(
                "search_web",
                {"query": query}
            )
            return response





In [125]:
query = "Who won the FIFA World Cup in 2026?"
result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL)
print(result)



  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/2293778172.py", line 2, in <module>
  |     result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL)
  |              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/2217194424.py", line 12, in run_mcp_query
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |   

In [117]:
MCP_SERVER_PATH = os.path.abspath("./mcp_servers/mcp_tools_server.py")

Lets first try to use the `web_search` tool

In [118]:
MCP_SERVER_PATH = sys.executable
MCP_SERVER_ARGS = [
    "/Users/jopanda/bootcamp-aipm/ds-mcp/mcp_servers/mcp_tools_server.py"
]
server_params = StdioServerParameters(
    command=MCP_SERVER_PATH,
    args=MCP_SERVER_ARGS
)


In [126]:
# Try a web search:
query = "Who won the FIFA World Cup in 2026?"
print(await run_mcp_query(MCP_SERVER_PATH,query, SMALL_MODEL))

# full working version:
async def run_mcp_query(server_path, query, model):
    server_params = StdioServerParameters(
        command=server_path,
        args=MCP_SERVER_ARGS
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            response = await session.call_tool(
                "web-search",
                {"query": query, "model": model}
            )
            return response


  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/3971358616.py", line 3, in <module>
  |     print(await run_mcp_query(MCP_SERVER_PATH,query, SMALL_MODEL))
  |           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/2217194424.py", line 12, in run_mcp_query
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |     async

In [ ]:
query = "Who won the FIFA World Cup in 2026?"
print(await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL))

  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1390156395.py", line 2, in <module>
  |     print(await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL))
  |           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1935980309.py", line 6, in run_mcp_query
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |     asyn

Now lets use the `read_local_file` tool

In [ ]:
# Try reading a file
query = "Summarize the contents of ai_response.txt"
pprint(await run_mcp_query(MCP_SERVER_PATH, query, SMALL_MODEL))

  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1660067009.py", line 3, in <module>
  |     pprint(await run_mcp_query(MCP_SERVER_PATH, query, SMALL_MODEL))
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1935980309.py", line 6, in run_mcp_query
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |     as

Try to run the above code block again if facing errors or weird outputs

The `SMALL_MODEL`, that is `llama3.2:1b` does not perform very well to even not so difficult tasks

In [129]:
async def run_mcp_query(python_interpreter, user_prompt, model, server_script):
    server_params = StdioServerParameters(
        command=python_interpreter,
        args=[server_script]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:

            await session.initialize()
            tools = await session.list_tools()

            # Convert MCP tools → Ollama tool format
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in tools.tools
            ]

            # Ask LLM what to do
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": user_prompt}],
                tools=ollama_tools,
            )

            # If LLM calls a tool
            if response.get("message", {}).get("tool_calls"):
                for tool_call in response["message"]["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    tool_args = tool_call["function"]["arguments"]

                    print(f"🛠️ Calling tool: {tool_name} with {tool_args}")

                    result = await session.call_tool(tool_name, tool_args)

                    # Final answer
                    final = ollama.chat(
                        model=model,
                        messages=[
                            {"role": "user", "content": user_prompt},
                            response["message"],
                            {
                                "role": "tool",
                                "content": str(result.content),
                                "name": tool_name,
                            },
                        ],
                    )
                    return final["message"]["content"]

            return response["message"]["content"]


In [131]:
query = "Summarize the contents of ai_response.txt"
result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL, MCP_TOOLS_SERVER)
pprint(result)


[02/23/26 14:10:50] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=545290;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=202428;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

🛠️ Calling tool: read_local_file with {'file_path': 'ai_response.txt'}


[02/23/26 14:10:54] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=756671;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=516279;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

('The contents of ai_response.txt appear to be a text file containing an '
 'explanation about AI Project Managers. The text is written in a friendly and '
 'approachable tone, using analogies such as building with Legos to help '
 'illustrate the role of an AI Project Manager.\n'
 '\n'
 'The text covers the key responsibilities of an AI Project Manager, including '
 'planning, coordination, problem-solving, and ensuring quality control. It '
 'also highlights the use of tools and technology to support project '
 'management tasks.\n'
 '\n'
 'Overall, the content seems to be educational in nature, aiming to explain '
 'the concept of an AI Project Manager in a way that is easy to understand.')


### Add System Prompts 

In [136]:
# my working code after debugging:
async def mcp_with_system_prompt(python_interpreter, user_prompt, model, server_script):
    
    server_params = StdioServerParameters(
        command=python_interpreter,     # python interpreter
        args=[server_script],           # the MCP server script
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List tools and format for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools_result.tools]

            # 1. Ask Ollama
            response = ollama.chat(
                model=model,
                messages=[
                    {
                        'role': 'system', 
                        'content': (
                            "You are a real-time assistant with access to tools. "
                            "Always prefer tool results over internal knowledge. "
                            "The current year is 2026."
                        )
                    },
                    {'role': 'user', 'content': user_prompt}
                ],
                tools=ollama_tools,
            )

            message = response.get('message', {})

            # 2. Check for tool calls
            if message.get('tool_calls'):
                tool_messages = []
                for tool_call in message['tool_calls']:
                    name = tool_call['function']['name']
                    args = tool_call['function']['arguments']
                    
                    print(f"🛠️ LLM decided to use: {name}")
                    result = await session.call_tool(name, args)
                    
                    tool_messages.append({
                        'role': 'tool',
                        'content': str(result.content),
                        'name': name
                    })
                
                # 3. Final synthesis
                final_response = ollama.chat(
                    model=model,
                    messages=[
                        {
                            'role': 'system', 
                            'content': (
                                "Use the tool results to answer the user's question. "
                                "Do not rely on training data if tool results exist."
                            )
                        },
                        {'role': 'user', 'content': user_prompt},
                        message,
                        *tool_messages
                    ]
                )
                return final_response['message']['content']
            
            return message.get('content', '')


In [137]:
# and my correct call:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(
    MCP_SERVER_PATH, 
    query, 
    SMALL_MODEL, 
    MCP_TOOLS_SERVER
))


[02/23/26 14:15:31] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=903699;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=195770;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

('{"name": "fifa world cup winner", "parameters": {"query": "fifa world cup '
 '2026 winner"}}')


In [138]:
# then:
pprint(await mcp_with_system_prompt(
    MCP_SERVER_PATH, 
    query, 
    LARGE_MODEL, 
    MCP_TOOLS_SERVER
))


[02/23/26 14:15:42] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=942578;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=841601;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

🛠️ LLM decided to use: search_web


[02/23/26 14:15:45] INFO     HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"   ]8;id=724426;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=218230;file:///Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\

('The 2026 FIFA World Cup has not yet taken place, so there is no winner to '
 'report. The tournament is scheduled to take place in the United States, '
 'Canada, and Mexico in 2026.')


So far so good, let's wrap up here and jump to the next notebook

Now lets add some system prompts and try to see if that makes a difference

In [133]:
async def mcp_with_system_prompt(mcp_server_path, user_prompt, model):
    
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List tools and format for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools_result.tools]

            # 1. Ask Ollama
            response = ollama.chat(
                model=model,
                messages=[
                    # --- ADDED THIS SYSTEM ROLE ---
                    {
                        'role': 'system', 
                        'content': (
                            "You are a real-time assistant with access to tools. "
                            "IMPORTANT: Always prefer information from tool results over your internal knowledge. "
                            "The current year is 2026. If tool data contradicts your training, the tool is right."
                        )
                    },
                    {'role': 'user', 'content': user_prompt}
                ],
                tools=ollama_tools,
            )

            # 2. Check for tool calls
            message = response.get('message', {})
            # print(f"LLM Response: {message.get('content', '')}")
            if message.get('tool_calls'):
                # Handle all tool calls (Ollama might suggest more than one)
                tool_messages = []
                for tool_call in message['tool_calls']:
                    name = tool_call['function']['name']
                    args = tool_call['function']['arguments']
                    
                    print(f"🛠️ LLM decided to use: {name}")
                    result = await session.call_tool(name, args)
                    
                    tool_messages.append({
                        'role': 'tool',
                        'content': str(result.content),
                        'name': name
                    })
                
                # 3. Final synthesis
                final_response = ollama.chat(
                    model=model,
                    messages=[
                        {
                            'role': 'system', 'content': (
                                "Summarize the tool results accurately only when necessary." 
                                "Do not rely on your training data but use the tools to get up-to-date information." 
                                "Always prefer tool results over your internal knowledge. "
                                )
                        },
                        {'role': 'user', 'content': user_prompt},
                        message,
                        *tool_messages
                    ]
                )
                return final_response['message']['content']
            
            return message['content']



Now, lets try again with both MODELS one by one

In [134]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH, query, SMALL_MODEL))

  + Exception Group Traceback (most recent call last):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/2635062028.py", line 2, in <module>
  |     pprint(await mcp_with_system_prompt(MCP_SERVER_PATH, query, SMALL_MODEL))
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/hf/d5g_dld107qb4pqtcyhnpb340000gn/T/ipykernel_69078/1325641094.py", line 8, in mcp_with_system_prompt
  |     async with stdio_client(server_params) as (read, write):
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/contextlib.py", line 222, in __aexit__
  |     await self.gen.athrow(typ, value, traceback)
  |   File "/Users/jopanda/.pyenv/versions/3.11.3/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 182,

In [ ]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH, query, LARGE_MODEL))

In [ ]:
query = "Summarize the contents of the text file ./ai_response.txt"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH,query, LARGE_MODEL))

If you are getting json responses try changing system prompt. 

<details>
    <summary>Try adding this (check only if needed)</summary>
    "You are an MCP Assistant. If you need information, use a tool. DO NOT write raw JSON in your response. If you call a tool, use the proper tool-call format."
</details>

### Getting simple python code from web search

In [ ]:
query = "Search for python code for calculating the first 10 numbers of the Fibonacci sequence."
print(await mcp_with_system_prompt(MCP_SERVER_PATH, query, LARGE_MODEL))

Copy paste the above code in a python file and name it as `fibonacci_numbers.py`. Create a folder `scripts` and move the python script under the `scripts` folder

Try running this python script in the terminal to check if the script is working or not

```bash
python scripts/fibonacci_numbers.py
```